Pipeline

In [1]:

# =============================================================================
# Instalação dos pacotes
# =============================================================================

!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install matplotlib
!pip install ultralytics
!pip install roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [2]:
# =============================================================================
# Importando as bibliotecas
# =============================================================================
import os
import shutil
import glob
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO #(YOLOv11)
from roboflow import Roboflow
matplotlib.use("Agg")

# =============================================================================
# Configuracoes globais
# =============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
API_KEY = "Md5T3krCkUBImQf9Df53"
WORKSPACE = "rapis-soft-technologies"
PROJECT = "harsh-0i6vw"
VERSION = 1

# Diretorios de trabalho
BASE_DIR = Path(".").resolve()
DOWNLOAD_DIR = BASE_DIR / "harsh_dataset"
YOLO_DIR = BASE_DIR / "yolo_harsh"
SPLIT_DIR = BASE_DIR / "splits"
RESULTS_DIR = BASE_DIR / "resultados"

YOLO_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Classes do dataset
CLASSES = ["bed", "coal"]
CLASS_NAMES = {0: "bed", 1: "coal"}


# def baixar_dataset() -> str:


# #  Realizar a leitura da base de dadados:
# #         0 = Vídeo | 1 = Imagem
# def load_dataset(dataset_path: str, data_type: int)-> np.ndarray:



# # Realiza o pre-processamento no dataset
# def preprocess(dataset: np.ndarray)-> np.ndarray:


# # Realiza a divisão do dataset: treinamento e teste e define a porcentagem de dados para teste escala entre [0 - 1]
# def split_dataset(dataset: np.ndarray,test_size: float = 0.2) -> tuple[np.ndarray, np.ndarray]):


# # Realizar o treinamento do modelo
# def train_model(model: str, train_dataset:np.ndarray,  test_dataset:np.ndarray]):-> np.ndarray


# # Realiza a análise do modelo através das métricas de avaliação de desempenho
# def evaluate_model(results:np.ndarray, test_dataset:np.ndarray):



# if __name__ == "__main__":
#   # Execução das funções.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# =============================================================================
# 1. DOWNLOAD AUTOMATICO DO ROBOFLOW
# =============================================================================

def load_dataset()-> tuple[Path, Path]:
    # Baixa o dataset 'harsh' do Roboflow
    print("\n[1] Baixando dataset do Roboflow...")
    rf = Roboflow(api_key=API_KEY)
    projeto = rf.workspace(WORKSPACE).project(PROJECT)
    versao = projeto.version(VERSION)
    dataset = versao.download("yolov11", location=str(DOWNLOAD_DIR))

    # O Roboflow cria uma pasta com o nome do dataset; localizamos o data.yaml
    data_yaml = None
    for p in DOWNLOAD_DIR.rglob("data.yaml"):
        data_yaml = p
        break

    if data_yaml is None:
        raise FileNotFoundError("data.yaml nao encontrado apos o download.")

    raiz_dataset = data_yaml.parent
    print(f"Dataset baixado em: {raiz_dataset}")

    # Contar imagens por split original
    splits_encontrados = {}
    for split_name in ["train", "valid", "test", "val"]:
        img_dir = raiz_dataset / split_name / "images"
        if img_dir.exists():
            n = len(list(img_dir.glob("*.*")))
            splits_encontrados[split_name] = n
            print(f"  Split '{split_name}': {n} imagens")

    total = sum(splits_encontrados.values())
    print(f"  TOTAL de imagens baixadas: {total}")

    return raiz_dataset, data_yaml


raiz_dataset, data_yaml_original = load_dataset()



[1] Baixando dataset do Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Dataset baixado em: /content/harsh_dataset
  Split 'train': 351 imagens
  Split 'valid': 28 imagens
  TOTAL de imagens baixadas: 379


In [5]:
# =============================================================================
# 2. PRE-PROCESSAMENTO DO DATASET
# =============================================================================

def preprocess(raiz_dataset: Path)-> np.ndarray:

    print("\n[2] Coletando todas as amostras (ignorando splits originais)...")
    registros = []

    # Procurar imagens em qualquer subpasta de imagens
    padroes_img = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"]
    for split_name in ["train", "valid", "test", "val"]:
        img_dir = raiz_dataset / split_name / "images"
        lbl_dir = raiz_dataset / split_name / "labels"
        if not img_dir.exists():
            continue

        for padrao in padroes_img:
            for img_path in img_dir.glob(padrao):
                stem = img_path.stem
                label_path = lbl_dir / f"{stem}.txt"

                # Descobrir classes presentes no label
                classes_presentes = set()
                if label_path.exists():
                    with open(label_path, "r", encoding="utf-8") as f:
                        for linha in f:
                            linha = linha.strip()
                            if linha:
                                partes = linha.split()
                                if partes and partes[0].isdigit():
                                    classes_presentes.add(int(partes[0]))

                # Chave de estratificacao: composicao de classes ordenada
                strat_key = "_".join(sorted([str(c) for c in classes_presentes]))
                if not strat_key:
                    strat_key = "sem_classe"

                registros.append({
                    "image_path": str(img_path),
                    "label_path": str(label_path),
                    "split_original": split_name,
                    "classes_presentes": sorted(classes_presentes),
                    "n_classes": len(classes_presentes),
                    "strat_key": strat_key,
                })

    df = pd.DataFrame(registros)
    print(f"  Total de amostras coletadas: {len(df)}")
    print(f"  Distribuicao por split original:\n{df['split_original'].value_counts().to_string()}")
    print(f"  Distribuicao por composicao de classes:\n{df['strat_key'].value_counts().to_string()}")
    return df


df = preprocess(raiz_dataset)


[2] Coletando todas as amostras (ignorando splits originais)...
  Total de amostras coletadas: 379
  Distribuicao por split original:
split_original
train    351
valid     28
  Distribuicao por composicao de classes:
strat_key
0      241
0_1     98
1       40


In [6]:
# =============================================================================
# 3. DIVISÃO DA BASE EM TREINAMENTO E TESTE
# =============================================================================

def split_dataset(df: np.ndarray) -> tuple[np.ndarray, np.ndarray]:

    print("\n[3] Realizando split estratificado 80/20 com numpy...")
    rng = np.random.RandomState(SEED)
    train_idx = []
    test_idx = []

    # Agrupar por chave de estratificacao
    for strat_key, grupo in df.groupby("strat_key"):
        indices = grupo.index.to_numpy()
        # Embaralhar com numpy
        rng.shuffle(indices)
        n_total = len(indices)
        n_train = int(np.round(0.8 * n_total))
        # Garantir que o teste tenha ao menos 1 amostra quando possivel
        if n_total > 1 and n_train == n_total:
            n_train = n_total - 1
        if n_train < 0:
            n_train = 0

        train_idx.extend(indices[:n_train].tolist())
        test_idx.extend(indices[n_train:].tolist())

    # Converter para arrays numpy e embaralhar novamente (ordem final)
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    df_train = df.loc[train_idx].reset_index(drop=True)
    df_test = df.loc[test_idx].reset_index(drop=True)

    print(f"  Treino: {len(df_train)} amostras ({100*len(df_train)/len(df):.1f}%)")
    print(f"  Teste:  {len(df_test)} amostras ({100*len(df_test)/len(df):.1f}%)")

    # Salvar train.txt e test.txt com caminhos das imagens
    train_txt = SPLIT_DIR / "train.txt"
    test_txt = SPLIT_DIR / "test.txt"
    df_train["image_path"].to_csv(train_txt, index=False, header=False)
    df_test["image_path"].to_csv(test_txt, index=False, header=False)
    print(f"  Arquivos salvos: {train_txt.name}, {test_txt.name}")

    return df_train, df_test

df_train, df_test = split_dataset(df)



[3] Realizando split estratificado 80/20 com numpy...
  Treino: 303 amostras (79.9%)
  Teste:  76 amostras (20.1%)
  Arquivos salvos: train.txt, test.txt


In [10]:
# =============================================================================
# 4. PREPARAR ESTRUTURA YOLO
# =============================================================================

def model_structure_configuration(df_train: np.ndarray, df_test:np.ndarray)-> Path:
    print("\n[4] Preparando estrutura de diretorios ...")

    for split_name, df_split in [("train", df_train), ("test", df_test)]:
        img_out = YOLO_DIR / "images" / split_name
        lbl_out = YOLO_DIR / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        for _, row in df_split.iterrows():
            src_img = Path(row["image_path"])
            src_lbl = Path(row["label_path"])

            if not src_img.exists():
                continue
            dst_img = img_out / src_img.name
            if not dst_img.exists():
                shutil.copy2(src_img, dst_img)

            if src_lbl.exists():
                dst_lbl = lbl_out / src_lbl.name
                if not dst_lbl.exists():
                    shutil.copy2(src_lbl, dst_lbl)
            else:
                dst_lbl = lbl_out / f"{src_img.stem}.txt"
                if not dst_lbl.exists():
                    dst_lbl.touch()

        n_imgs = len(list(img_out.glob("*.*")))
        n_lbls = len(list(lbl_out.glob("*.txt")))
        print(f"  {split_name}: {n_imgs} imagens, {n_lbls} labels")

    # Criar data.yaml
    data_yaml_path = YOLO_DIR / "data.yaml"
    with open(data_yaml_path, "w", encoding="utf-8") as f:
        f.write(f"path: {YOLO_DIR.as_posix()}\n")
        f.write("train: images/train\n")
        f.write("val: images/test\n")
        f.write("test: images/test\n")
        f.write(f"nc: {len(CLASSES)}\n")
        f.write(f"names: {CLASSES}\n")
    print(f"  data.yaml criado em: {data_yaml_path}")
    return data_yaml_path


data_yaml_path = model_structure_configuration(df_train, df_test)




[4] Preparando estrutura de diretorios ...
  train: 303 imagens, 303 labels
  test: 76 imagens, 76 labels
  data.yaml criado em: /content/yolo_harsh/data.yaml


In [13]:
# =============================================================================
# 5. CONFIGURAÇÃO DO MODELO
# =============================================================================
def train_model(data_yaml_path: Path)-> Path:

    print("\n[5] Iniciando treinamento ...")
    print("\n[4] Iniciando treinamento YOLOv11 (segmentacao)...")

    # Tentar yolov11m-seg.pt; se nao existir, usar yolov11n-seg.pt
    # Nota: ultralytics tambem aceita nomes como yolo11m-seg.pt
    modelos_candidatos = [
        "yolov11m-seg.pt",
        "yolo11m-seg.pt",
        "yolov11n-seg.pt",
        "yolo11n-seg.pt",
    ]

    modelo_escolhido = None
    for nome in modelos_candidatos:
        try:
            print(f"  Tentando carregar: {nome}")
            modelo = YOLO(nome)
            modelo_escolhido = nome
            print(f"  Modelo carregado com sucesso: {nome}")
            break
        except Exception as e:
            print(f"  Falha ao carregar {nome}: {e}")

    if modelo_escolhido is None:
        raise RuntimeError("Nao foi possivel carregar nenhum modelo YOLOv11 de segmentacao.")

    resultados = modelo.train(
        data=str(data_yaml_path),
        epochs=50,
        batch=8,
        imgsz=640,
        seed=SEED,
        project=str(RESULTS_DIR),
        name="treino_yolov11_harsh",
        exist_ok=True,
        verbose=True,
    )

    # Caminho do melhor modelo
    best_weights = RESULTS_DIR / "treino_yolov11_harsh" / "weights" / "best.pt"
    print(f"  Treinamento concluido. Melhor modelo: {best_weights}")
    return best_weights










best_weights = train_model(data_yaml_path)


[5] Iniciando treinamento ...

[4] Iniciando treinamento YOLOv11 (segmentacao)...
  Tentando carregar: yolov11m-seg.pt
  Falha ao carregar yolov11m-seg.pt: [Errno 2] No such file or directory: 'yolov11m-seg.pt'
  Tentando carregar: yolo11m-seg.pt
  Modelo carregado com sucesso: yolo11m-seg.pt
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_harsh/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

In [15]:
# =============================================================================
# 6. AVALIACAO ESTATISTICA
# =============================================================================

def evaluate_model(best_weights:Path, data_yaml_path:Path)->tuple[np.ndarray, np.ndarray]:

    print("\n[6] Avaliacao estatistica no conjunto de teste...")
    modelo = YOLO(str(best_weights))

    metricas = modelo.val(
        data=str(data_yaml_path),
        split="test",
        seed=SEED,
        project=str(RESULTS_DIR),
        name="avaliacao_yolov11_harsh",
        exist_ok=True,
        verbose=True,
        plots=True,
        save_json=True,
    )

    # ---- Extrair metricas globais ----
    box = metricas.box
    precision_global = float(box.mp)
    recall_global = float(box.mr)
    map50_global = float(box.map50)
    map5095_global = float(box.map)

    print("\n--- Metricas Globais ---")
    print(f"Precision global: {precision_global:.4f}")
    print(f"Recall global:    {recall_global:.4f}")
    print(f"mAP50 global:     {map50_global:.4f}")
    print(f"mAP50-95 global:  {map5095_global:.4f}")

    # ---- Metricas por classe ----
    n_classes = len(CLASSES)
    p_por_classe = np.array(box.p, dtype=float).reshape(-1)[:n_classes]
    r_por_classe = np.array(box.r, dtype=float).reshape(-1)[:n_classes]
    map50_por_classe = np.array(box.ap50, dtype=float).reshape(-1)[:n_classes]
    map5095_por_classe = np.array(box.ap, dtype=float).reshape(-1)[:n_classes]

    # Garantir tamanho correto (caso venha em formato diferente)
    def ajustar(arr, n):
        arr = np.array(arr, dtype=float).reshape(-1)
        if len(arr) < n:
            arr = np.pad(arr, (0, n - len(arr)), constant_values=np.nan)
        return arr[:n]

    p_por_classe = ajustar(p_por_classe, n_classes)
    r_por_classe = ajustar(r_por_classe, n_classes)
    map50_por_classe = ajustar(map50_por_classe, n_classes)
    map5095_por_classe = ajustar(map5095_por_classe, n_classes)

    # F1 por classe
    f1_por_classe = np.where(
        (p_por_classe + r_por_classe) > 0,
        2 * (p_por_classe * r_por_classe) / (p_por_classe + r_por_classe + 1e-12),
        0.0,
    )
    f1_macro = float(np.nanmean(f1_por_classe))

    # ---- DataFrame de metricas por classe ----
    df_metricas = pd.DataFrame({
        "classe": CLASSES,
        "precision": p_por_classe,
        "recall": r_por_classe,
        "f1": f1_por_classe,
        "mAP50": map50_por_classe,
        "mAP50-95": map5095_por_classe,
    })

    print("\n--- Metricas por Classe ---")
    print(df_metricas.to_string(index=False))
    print(f"\nF1-Score macro medio: {f1_macro:.4f}")

    # ---- Estatisticas: media, desvio padrao e IC 95% (t de Student) ----
    colunas_metricas = ["precision", "recall", "f1", "mAP50", "mAP50-95"]
    linhas_stats = []

    n = n_classes
    graus_liberdade = max(n - 1, 1)
    t_crit = stats.t.ppf(0.975, graus_liberdade)  # bilateral 95%

    for col in colunas_metricas:
        valores = df_metricas[col].to_numpy(dtype=float)
        media = float(np.nanmean(valores))
        desvio = float(np.nanstd(valores, ddof=1)) if n > 1 else 0.0
        erro_padrao = desvio / np.sqrt(n) if n > 0 else 0.0
        ic_inf = media - t_crit * erro_padrao
        ic_sup = media + t_crit * erro_padrao
        linhas_stats.append({
            "metrica": col,
            "media": media,
            "desvio_padrao": desvio,
            "ic95_inf": ic_inf,
            "ic95_sup": ic_sup,
            "t_critico": t_crit,
        })

    df_stats = pd.DataFrame(linhas_stats)
    print("\n--- Estatisticas (media, desvio padrao, IC 95%) ---")
    print(df_stats.to_string(index=False))

    # ---- Matriz de confusao com seaborn ----
    matriz_confusao = None
    # Procurar arquivo de matriz de confusao gerado pelo val
    aval_dir = RESULTS_DIR / "avaliacao_yolov11_harsh"
    arquivos_cm = list(aval_dir.glob("*confusion_matrix*"))
    if arquivos_cm:
        print(f"\nMatriz de confusao gerada pelo ultralytics: {arquivos_cm[0]}")
    else:
        print("\nMatriz de confusao nao encontrada nos arquivos do ultralytics.")

    # Tentar obter matriz de confusao do objeto metricas (se disponivel)
    try:
        mc = getattr(metricas, "confusion_matrix", None)
        if mc is not None and hasattr(mc, "matrix"):
            matriz_confusao = np.array(mc.matrix)
            # Plotar com seaborn
            plt.figure(figsize=(8, 6))
            labels_plot = CLASSES + ["background"] if matriz_confusao.shape[0] == len(CLASSES) + 1 else CLASSES
            sns.heatmap(
                matriz_confusao,
                annot=True,
                fmt=".0f",
                cmap="Blues",
                xticklabels=labels_plot,
                yticklabels=labels_plot,
            )
            plt.title("Matriz de Confusao - YOLOv11 (harsh)")
            plt.xlabel("Predito")
            plt.ylabel("Real")
            plt.tight_layout()
            cm_path = RESULTS_DIR / "matriz_confusao_seaborn.png"
            plt.savefig(cm_path, dpi=150)
            plt.close()
            print(f"Matriz de confusao (seaborn) salva em: {cm_path}")
    except Exception as e:
        print(f"Nao foi possivel plotar matriz de confusao via seaborn: {e}")

    # ---- Boxplot comparativo das metricas por classe ----
    df_box = df_metricas.melt(
        id_vars=["classe"],
        value_vars=colunas_metricas,
        var_name="metrica",
        value_name="valor",
    )
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_box, x="metrica", y="valor", palette="Set2")
    sns.stripplot(data=df_box, x="metrica", y="valor", color="black", size=5, jitter=True)
    plt.title("Boxplot Comparativo das Metricas por Classe - YOLOv11 (harsh)")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    box_path = RESULTS_DIR / "boxplot_metricas_por_classe.png"
    plt.savefig(box_path, dpi=150)
    plt.close()
    print(f"Boxplot das metricas salvo em: {box_path}")

    # ---- Salvar relatorio CSV ----
    relatorio_path = RESULTS_DIR / "relatorio_metricas.csv"
    df_metricas.to_csv(relatorio_path, index=False, encoding="utf-8-sig")
    print(f"Relatorio de metricas salvo em: {relatorio_path}")

    df_stats_path = RESULTS_DIR / "relatorio_estatisticas.csv"
    df_stats.to_csv(df_stats_path, index=False, encoding="utf-8-sig")
    print(f"Estatisticas (media/desvio/IC95) salvas em: {df_stats_path}")

    # ---- Relatorio final detalhado no terminal ----
    print("\n" + "=" * 70)
    print("RELATORIO FINAL DE AVALIACAO")
    print("=" * 70)
    print(f"Classes: {CLASSES}")
    print(f"Numero de classes: {n_classes}")
    print(f"Seed utilizada: {SEED}")
    print("-" * 70)
    print("METRICAS GLOBAIS:")
    print(f"  Precision global : {precision_global:.4f}")
    print(f"  Recall global    : {recall_global:.4f}")
    print(f"  mAP50 global     : {map50_global:.4f}")
    print(f"  mAP50-95 global  : {map5095_global:.4f}")
    print(f"  F1 macro medio   : {f1_macro:.4f}")
    print("-" * 70)
    print("METRICAS POR CLASSE:")
    print(df_metricas.to_string(index=False))
    print("-" * 70)
    print("ESTATISTICAS (media, desvio padrao, IC 95% - t de Student):")
    print(df_stats.to_string(index=False))
    print("-" * 70)
    print(f"Arquivos gerados em: {RESULTS_DIR}")
    print("=" * 70)

    return df_metricas, df_stats


df_metricas, df_stats = evaluate_model(best_weights, data_yaml_path)

print("\nProcesso concluido com sucesso!")



[6] Avaliacao estatistica no conjunto de teste...
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m-seg summary (fused): 139 layers, 22,336,854 parameters, 0 gradients, 112.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2509.2±840.5 MB/s, size: 333.6 KB)
val: Scanning /content/yolo_harsh/labels/test.cache... 76 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 76/76 26.6Mit/s 0.0s
requirements: Ultralytics requirement ['faster-coco-eval>=1.6.7'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 544ms
Prepared 1 package in 55ms
Installed 1 package in 4ms
 + faster-coco-eval==1.7.2

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.0it/s 5.0s
          



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



Boxplot das metricas salvo em: /content/resultados/boxplot_metricas_por_classe.png
Relatorio de metricas salvo em: /content/resultados/relatorio_metricas.csv
Estatisticas (media/desvio/IC95) salvas em: /content/resultados/relatorio_estatisticas.csv

RELATORIO FINAL DE AVALIACAO
Classes: ['bed', 'coal']
Numero de classes: 2
Seed utilizada: 42
----------------------------------------------------------------------
METRICAS GLOBAIS:
  Precision global : 0.9971
  Recall global    : 0.9741
  mAP50 global     : 0.9946
  mAP50-95 global  : 0.9405
  F1 macro medio   : 0.9853
----------------------------------------------------------------------
METRICAS POR CLASSE:
classe  precision  recall       f1    mAP50  mAP50-95
   bed   1.000000 0.94827 0.973448 0.994178  0.940737
  coal   0.994121 1.00000 0.997052 0.995000  0.940208
----------------------------------------------------------------------
ESTATISTICAS (media, desvio padrao, IC 95% - t de Student):
  metrica    media  desvio_padrao  ic95_in